# NB09A — Row-Level Leakage Diagnostic

**Purpose.** Quantify, on the same SCiO shell-egg dataset, how predictive performance changes when repeated spectra are split randomly at the **row level** instead of keeping eggs disjoint.

This notebook is a **review-driven diagnostic analysis**. It does **not** replace the frozen NB01–NB08 results and does not alter the primary egg-disjoint estimates.

## Design principles

- The biological unit remains the egg (`sample`).
- The valid reference is the frozen egg-disjoint OOF analysis.
- The diagnostic condition intentionally allows spectra from the same egg to occur in both training and test folds.
- To isolate the effect of the partitioning strategy, model architecture and representative configurations are **frozen from the leakage-safe analysis before this notebook is executed**.
- No diagnostic row-level result is used to retune or replace the primary manuscript models.
- Deep-learning seeds: 2026, 2027, 2028.
- Primary comparison: MAE, RMSE, R² and optimism gap between row-level CV and egg-disjoint OOF.

In [ ]:
# 1. Mount Drive and define project paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, time, random, hashlib, warnings
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026')
RAW = ROOT / '01_DATA_RAW' / 'dataset_egg_storage_RAW.csv'
NB03_DIR = ROOT / '05_RESULTS' / 'NB03_CHEMOMETRIC_BASELINES'
NB04_DIR = ROOT / '05_RESULTS' / 'NB04_DEEP_LEARNING_BENCHMARK'
NB07_DIR = ROOT / '05_RESULTS' / 'NB07_PRACTICAL_APPLICABILITY'
OUT_DIR = ROOT / '05_RESULTS' / 'NB09A_ROW_LEVEL_LEAKAGE_DIAGNOSTIC'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DATASET_SHA256 = 'cd5021c555ae6b57f892549c574599cef75edf87f58b3f7f4d246ade9327d15e'
RUN_REVISION = 'NB09A_v1_review_rowlevel_diagnostic'

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026
OUT_DIR: /content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026/05_RESULTS/NB09A_ROW_LEVEL_LEAKAGE_DIAGNOSTIC


In [ ]:
# 2. Environment and GPU gate
import sklearn, scipy
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('scipy:', scipy.__version__)

import tensorflow as tf
print('tensorflow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpus)

if not gpus:
    warnings.warn(
        'No GPU detected. Chemometric models will run normally, but the 60 deep-learning fits '
        'may be slow on CPU. In Colab use Runtime > Change runtime type > GPU.'
    )

numpy: 2.1.3
pandas: 2.2.3
scikit-learn: 1.6.1
scipy: 1.16.3
tensorflow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# 3. Integrity audit of the raw dataset
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

observed_hash = sha256_file(RAW)
assert observed_hash == EXPECTED_DATASET_SHA256, (observed_hash, EXPECTED_DATASET_SHA256)

df = pd.read_csv(RAW)
spec_cols = [c for c in df.columns if c.startswith('Spectra_')]
spec_cols = sorted(spec_cols, key=lambda x: int(x.split('_')[1]))

assert df.shape == (660, 333)
assert df['sample'].nunique() == 30
assert df['storage_days'].nunique() == 22
assert len(spec_cols) == 331
assert df.groupby(['sample','storage_days']).size().eq(1).all()

X = df[spec_cols].to_numpy(np.float64)
y = df['storage_days'].to_numpy(np.float64)
samples = df['sample'].to_numpy()

print('PASS — raw dataset and repeated-measures structure verified.')

PASS — raw dataset and repeated-measures structure verified.


## Frozen representative configurations

The goal is to isolate **partitioning**, not to let the deliberately leaky condition perform a new hyperparameter search. Representative configurations are therefore derived deterministically from NB03/NB04 selections:

- PLSR: modal preprocessing and median number of components.
- SVR: modal preprocessing and median \(C\), \(\epsilon\), \(\gamma\).
- Deep learning: modal preprocessing and median selected epoch for each architecture.
- Network capacity is identical to the frozen NB04 architecture.

The representative configuration is written to disk before model fitting.

In [ ]:
# 4. Derive frozen representative configurations from NB03/NB04
cfg03 = pd.read_csv(NB03_DIR / 'NB03_selected_configurations.csv')
cfg04 = pd.read_csv(NB04_DIR / 'NB04_selected_configurations.csv')

def deterministic_mode(series):
    vc = series.dropna().astype(str).value_counts()
    top = vc[vc == vc.max()].index.tolist()
    return sorted(top)[0]

rep = {}

p = cfg03[cfg03.model == 'PLSR']
rep['PLSR'] = {
    'preprocessing': deterministic_mode(p['preprocessing']),
    'n_components': int(round(float(p['n_components'].median())))
}

s = cfg03[cfg03.model == 'SVR']
rep['SVR'] = {
    'preprocessing': deterministic_mode(s['preprocessing']),
    'C': float(s['C'].median()),
    'epsilon': float(s['epsilon'].median()),
    'gamma': float(s['gamma'].median())
}

for model in ['ANN','SimpleRNN','LSTM','BiLSTM']:
    z = cfg04[cfg04.model == model]
    rep[model] = {
        'preprocessing': deterministic_mode(z['selected_preprocessing']),
        'epochs': int(round(float(z['selected_epoch'].median())))
    }

rep['DummyMean'] = {}
print(json.dumps(rep, indent=2))
(OUT_DIR / 'NB09A_representative_frozen_configurations.json').write_text(
    json.dumps(rep, indent=2), encoding='utf-8'
)

{
  "PLSR": {
    "preprocessing": "snv",
    "n_components": 40
  },
  "SVR": {
    "preprocessing": "sg_deriv1",
    "C": 100000.0,
    "epsilon": 2.0,
    "gamma": 1e-05
  },
  "ANN": {
    "preprocessing": "sg_deriv1",
    "epochs": 75
  },
  "SimpleRNN": {
    "preprocessing": "sg_deriv1",
    "epochs": 34
  },
  "LSTM": {
    "preprocessing": "sg_deriv1",
    "epochs": 76
  },
  "BiLSTM": {
    "preprocessing": "sg_deriv1",
    "epochs": 28
  },
  "DummyMean": {}
}


475

In [ ]:
# 5. Preprocessing — training-only fitted quantities
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler

class SpectralPreprocessor:
    def __init__(self, method, sg_window=11, sg_polyorder=2):
        self.method = str(method).lower()
        self.sg_window = int(sg_window)
        self.sg_polyorder = int(sg_polyorder)
        self.msc_reference_ = None
        self.scaler_ = None

    def _base_fit(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.method == 'msc':
            self.msc_reference_ = X.mean(axis=0)
        return self._base_transform(X)

    def _base_transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.method in {'raw','none'}:
            return X.copy()
        if self.method == 'snv':
            mu = X.mean(axis=1, keepdims=True)
            sd = X.std(axis=1, keepdims=True)
            sd[sd == 0] = 1.0
            return (X - mu) / sd
        if self.method == 'msc':
            if self.msc_reference_ is None:
                raise RuntimeError('MSC reference not fitted.')
            ref = self.msc_reference_
            refc = ref - ref.mean()
            den = np.sum(refc**2)
            xm = X.mean(axis=1, keepdims=True)
            slopes = np.sum((X - xm) * refc[None,:], axis=1) / den
            slopes[np.abs(slopes) < 1e-12] = 1.0
            intercepts = X.mean(axis=1) - slopes * ref.mean()
            return (X - intercepts[:,None]) / slopes[:,None]
        if self.method == 'sg_smooth':
            return savgol_filter(X, 11, 2, deriv=0, axis=1, mode='interp')
        if self.method == 'sg_deriv1':
            return savgol_filter(X, 11, 2, deriv=1, delta=1.0, axis=1, mode='interp')
        raise ValueError(self.method)

    def fit(self, X):
        base = self._base_fit(X)
        self.scaler_ = StandardScaler().fit(base)
        return self

    def transform(self, X):
        return self.scaler_.transform(self._base_transform(X))

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [ ]:
# 6. Metrics and model builders
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers

def reg_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    e = y_pred - y_true
    ae = np.abs(e)
    return {
        'MAE_days': float(mean_absolute_error(y_true, y_pred)),
        'RMSE_days': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': float(r2_score(y_true, y_pred)),
        'bias_days': float(e.mean()),
        'median_AE_days': float(np.median(ae)),
        'within_1d_pct': float((ae <= 1).mean()*100),
        'within_2d_pct': float((ae <= 2).mean()*100),
        'within_3d_pct': float((ae <= 3).mean()*100),
    }

def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def build_deep(model_name, n_features=331):
    if model_name == 'ANN':
        inp = keras.Input(shape=(n_features,))
        z = layers.Dense(64, activation='relu')(inp)
        z = layers.Dropout(0.20)(z)
        z = layers.Dense(32, activation='relu')(z)
        z = layers.Dropout(0.20)(z)
        out = layers.Dense(1)(z)
    else:
        inp = keras.Input(shape=(n_features,1))
        if model_name == 'SimpleRNN':
            z = layers.SimpleRNN(64)(inp)
        elif model_name == 'LSTM':
            z = layers.LSTM(64)(inp)
        elif model_name == 'BiLSTM':
            z = layers.Bidirectional(layers.LSTM(64))(inp)
        else:
            raise ValueError(model_name)
        z = layers.Dropout(0.20)(z)
        z = layers.Dense(32, activation='relu')(z)
        z = layers.Dropout(0.20)(z)
        out = layers.Dense(1)(z)
    m = keras.Model(inp, out, name=model_name)
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='mse',
        metrics=[keras.metrics.MeanAbsoluteError(name='mae')]
    )
    return m

## Diagnostic row-level partition

Five random K-fold splits are used over the **660 rows**. This intentionally violates egg independence. The notebook records the number of eggs shared between training and test in each fold so the leakage mechanism is directly visible.

In [ ]:
# 7. Frozen row-level folds and leakage audit
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=2026)
row_folds = list(kf.split(np.arange(len(df))))

leak_rows = []
for fold, (tr, te) in enumerate(row_folds, 1):
    train_eggs = set(samples[tr])
    test_eggs = set(samples[te])
    overlap = train_eggs & test_eggs
    leak_rows.append({
        'row_fold': fold,
        'n_train_rows': len(tr),
        'n_test_rows': len(te),
        'n_train_eggs': len(train_eggs),
        'n_test_eggs': len(test_eggs),
        'n_overlapping_eggs': len(overlap),
        'overlap_fraction_test_eggs': len(overlap)/len(test_eggs),
    })

leak_audit = pd.DataFrame(leak_rows)
display(leak_audit)
assert (leak_audit.n_overlapping_eggs > 0).all()
leak_audit.to_csv(OUT_DIR / 'NB09A_rowlevel_leakage_audit.csv', index=False)

,row_fold,n_train_rows,n_test_rows,n_train_eggs,n_test_eggs,n_overlapping_eggs,overlap_fraction_test_eggs
0,1,528,132,30,30,30,1.0
1,2,528,132,30,30,30,1.0
2,3,528,132,30,30,30,1.0
3,4,528,132,30,30,30,1.0
4,5,528,132,30,30,30,1.0


In [ ]:
# 8. Row-level CV: DummyMean, PLSR and SVR
from sklearn.preprocessing import StandardScaler as SKStandardScaler

pred_rows = []
fold_metrics = []

for fold, (tr, te) in enumerate(row_folds, 1):
    Xtr, Xte = X[tr], X[te]
    ytr, yte = y[tr], y[te]

    # DummyMean
    pred = np.full(len(te), ytr.mean(), dtype=float)
    for i, idx in enumerate(te):
        pred_rows.append({'row_index':int(idx),'sample':int(samples[idx]),'storage_days':float(y[idx]),
                          'row_fold':fold,'model':'DummyMean','seed':np.nan,'y_pred':float(pred[i])})
    fold_metrics.append({'row_fold':fold,'model':'DummyMean','seed':np.nan,**reg_metrics(yte,pred)})

    # PLSR
    c = rep['PLSR']
    pp = SpectralPreprocessor(c['preprocessing'])
    A = pp.fit_transform(Xtr)
    B = pp.transform(Xte)
    mdl = PLSRegression(n_components=c['n_components'], scale=True, max_iter=1000)
    mdl.fit(A, ytr)
    pred = mdl.predict(B).ravel()
    for i, idx in enumerate(te):
        pred_rows.append({'row_index':int(idx),'sample':int(samples[idx]),'storage_days':float(y[idx]),
                          'row_fold':fold,'model':'PLSR','seed':np.nan,'y_pred':float(pred[i])})
    fold_metrics.append({'row_fold':fold,'model':'PLSR','seed':np.nan,**reg_metrics(yte,pred)})

    # SVR — mirrors frozen NB03 double train-fitted scaling
    c = rep['SVR']
    pp = SpectralPreprocessor(c['preprocessing'])
    A0 = pp.fit_transform(Xtr)
    B0 = pp.transform(Xte)
    sc2 = SKStandardScaler().fit(A0)
    A, B = sc2.transform(A0), sc2.transform(B0)
    mdl = SVR(kernel='rbf', C=c['C'], epsilon=c['epsilon'], gamma=c['gamma'])
    mdl.fit(A, ytr)
    pred = mdl.predict(B)
    for i, idx in enumerate(te):
        pred_rows.append({'row_index':int(idx),'sample':int(samples[idx]),'storage_days':float(y[idx]),
                          'row_fold':fold,'model':'SVR','seed':np.nan,'y_pred':float(pred[i])})
    fold_metrics.append({'row_fold':fold,'model':'SVR','seed':np.nan,**reg_metrics(yte,pred)})

pd.DataFrame(pred_rows).to_csv(OUT_DIR / 'NB09A_rowlevel_oof_partial_chemometrics.csv', index=False)
pd.DataFrame(fold_metrics).to_csv(OUT_DIR / 'NB09A_rowlevel_fold_metrics_partial_chemometrics.csv', index=False)
print('PASS — chemometric diagnostic completed.')

PASS — chemometric diagnostic completed.


In [ ]:
# 9. Row-level CV: ANN / SimpleRNN / LSTM / BiLSTM
#    5 folds × 4 models × 3 seeds = 60 final fits.
#    Checkpoint is updated after every completed fit.

DL_MODELS = ['ANN','SimpleRNN','LSTM','BiLSTM']
SEEDS = [2026, 2027, 2028]
checkpoint_path = OUT_DIR / 'NB09A_DL_checkpoint.csv'

if checkpoint_path.exists():
    existing = pd.read_csv(checkpoint_path)
    dl_rows = existing.to_dict('records')
    done = set(zip(existing['row_fold'], existing['model'], existing['seed']))
    print('Resuming checkpoint:', len(done), 'fits already represented.')
else:
    dl_rows = []
    done = set()

for fold, (tr, te) in enumerate(row_folds, 1):
    Xtr, Xte = X[tr], X[te]
    ytr, yte = y[tr], y[te]

    for model_name in DL_MODELS:
        c = rep[model_name]
        pp = SpectralPreprocessor(c['preprocessing'])
        A = pp.fit_transform(Xtr).astype('float32')
        B = pp.transform(Xte).astype('float32')
        if model_name != 'ANN':
            A = A[...,None]
            B = B[...,None]

        for seed in SEEDS:
            key = (fold, model_name, seed)
            if key in done:
                continue

            tf.keras.backend.clear_session()
            set_all_seeds(seed)
            mdl = build_deep(model_name, n_features=331)

            t0 = time.time()
            mdl.fit(
                A, ytr.astype('float32'),
                epochs=int(c['epochs']),
                batch_size=32,
                verbose=0,
                shuffle=True
            )
            pred = mdl.predict(B, batch_size=32, verbose=0).reshape(-1)
            elapsed = time.time() - t0

            for i, idx in enumerate(te):
                dl_rows.append({
                    'row_index': int(idx),
                    'sample': int(samples[idx]),
                    'storage_days': float(y[idx]),
                    'row_fold': fold,
                    'model': model_name,
                    'seed': seed,
                    'y_pred': float(pred[i]),
                    'fit_seconds_for_fold_seed': float(elapsed)
                })

            pd.DataFrame(dl_rows).to_csv(checkpoint_path, index=False)
            done.add(key)
            print(f'PASS fold={fold} model={model_name} seed={seed} seconds={elapsed:.1f}')

print('PASS — all diagnostic DL fits completed.')

PASS fold=1 model=ANN seed=2026 seconds=11.3
PASS fold=1 model=ANN seed=2027 seconds=9.4


PASS fold=1 model=ANN seed=2028 seconds=9.3
PASS fold=1 model=SimpleRNN seed=2026 seconds=106.8
PASS fold=1 model=SimpleRNN seed=2027 seconds=105.8
PASS fold=1 model=SimpleRNN seed=2028 seconds=104.8
PASS fold=1 model=LSTM seed=2026 seconds=25.5
PASS fold=1 model=LSTM seed=2027 seconds=25.3
PASS fold=1 model=LSTM seed=2028 seconds=25.4
PASS fold=1 model=BiLSTM seed=2026 seconds=16.6
PASS fold=1 model=BiLSTM seed=2027 seconds=16.6
PASS fold=1 model=BiLSTM seed=2028 seconds=16.6
PASS fold=2 model=ANN seed=2026 seconds=9.5
PASS fold=2 model=ANN seed=2027 seconds=9.4
PASS fold=2 model=ANN seed=2028 seconds=9.5
PASS fold=2 model=SimpleRNN seed=2026 seconds=104.8
PASS fold=2 model=SimpleRNN seed=2027 seconds=105.1
PASS fold=2 model=SimpleRNN seed=2028 seconds=104.9
PASS fold=2 model=LSTM seed=2026 seconds=25.3
PASS fold=2 model=LSTM seed=2027 seconds=25.6
PASS fold=2 model=LSTM seed=2028 seconds=25.3
PASS fold=2 model=BiLSTM seed=2026 seconds=16.6
PASS fold=2 model=BiLSTM seed=2027 seconds=1

In [ ]:
# 10. Consolidate row-level OOF predictions and descriptive metrics
chem = pd.read_csv(OUT_DIR / 'NB09A_rowlevel_oof_partial_chemometrics.csv')
dl = pd.read_csv(checkpoint_path)

# Average the three DL predictions per row for descriptive comparison.
dl_mean = (dl.groupby(['row_index','sample','storage_days','row_fold','model'], as_index=False)
             .agg(y_pred=('y_pred','mean')))
row_oof = pd.concat([
    chem[['row_index','sample','storage_days','row_fold','model','y_pred']],
    dl_mean
], ignore_index=True)

row_oof.to_csv(OUT_DIR / 'NB09A_rowlevel_oof_seedmean.csv', index=False)
dl.to_csv(OUT_DIR / 'NB09A_rowlevel_oof_seedwise.csv', index=False)

summary = []
for model, g in row_oof.groupby('model'):
    summary.append({'model':model, **reg_metrics(g.storage_days, g.y_pred)})
row_summary = pd.DataFrame(summary).sort_values('MAE_days')
row_summary.to_csv(OUT_DIR / 'NB09A_rowlevel_performance_summary.csv', index=False)
display(row_summary)

,model,MAE_days,RMSE_days,R2,bias_days,median_AE_days,within_1d_pct,within_2d_pct,within_3d_pct
5,SVR,2.052577,2.540943,0.839593,-5.817162e-04,1.749166,30.000000,54.242424,75.454545
4,PLSR,2.104513,2.650513,0.825460,-3.097823e-02,1.791889,31.212121,55.606061,73.484848
0,ANN,2.173206,2.808225,0.804071,-3.866142e-01,1.706246,30.909091,57.878788,73.181818
1,BiLSTM,4.324740,5.264884,0.311329,-4.271353e-01,3.861511,13.484848,25.606061,38.787879
3,LSTM,4.529460,5.402700,0.274803,-4.166924e-01,4.240668,10.909091,24.696970,36.212121
6,SimpleRNN,4.829921,5.655501,0.205349,-4.656121e-01,4.672805,9.848485,20.757576,32.727273
2,DummyMean,5.509223,6.355293,-0.003472,4.306320e-17,5.532197,9.090909,18.181818,27.272727


In [ ]:
# 11. Compare diagnostic row-level CV with frozen valid egg-disjoint OOF
safe_path = NB07_DIR / 'NB07_operational_performance_summary.csv'
safe = pd.read_csv(safe_path)

# Normalize expected column names from NB07.
safe = safe.rename(columns={
    'MAE':'MAE_days','RMSE':'RMSE_days','R²':'R2',
    'MAE (days)':'MAE_days','RMSE (days)':'RMSE_days'
})
if 'R²' in safe.columns and 'R2' not in safe.columns:
    safe = safe.rename(columns={'R²':'R2'})
if 'model' not in safe.columns and 'Model' in safe.columns:
    safe = safe.rename(columns={'Model':'model'})

needed = ['model','MAE_days','RMSE_days','R2']
safe_small = safe[needed].copy()

cmp = safe_small.merge(
    row_summary[['model','MAE_days','RMSE_days','R2']],
    on='model', suffixes=('_eggdisjoint','_rowlevel')
)
cmp['MAE_optimism_gap_days'] = cmp['MAE_days_eggdisjoint'] - cmp['MAE_days_rowlevel']
cmp['RMSE_optimism_gap_days'] = cmp['RMSE_days_eggdisjoint'] - cmp['RMSE_days_rowlevel']
cmp['R2_inflation'] = cmp['R2_rowlevel'] - cmp['R2_eggdisjoint']
cmp['MAE_relative_reduction_pct'] = (
    100 * cmp['MAE_optimism_gap_days'] / cmp['MAE_days_eggdisjoint']
)

cmp = cmp.sort_values('MAE_days_eggdisjoint')
cmp.to_csv(OUT_DIR / 'NB09A_ROWLEVEL_VS_EGGDISJOINT_COMPARISON.csv', index=False)
display(cmp)

,model,MAE_days_eggdisjoint,RMSE_days_eggdisjoint,R2_eggdisjoint,MAE_days_rowlevel,RMSE_days_rowlevel,R2_rowlevel,MAE_optimism_gap_days,RMSE_optimism_gap_days,R2_inflation,MAE_relative_reduction_pct
0,SVR,2.194628,2.716171,0.816706,2.052577,2.540943,0.839593,0.142051,0.175228,0.022887,6.472660
1,PLSR,2.267298,2.863691,0.796255,2.104513,2.650513,0.825460,0.162785,0.213179,0.029205,7.179705
2,ANN,2.288940,2.974129,0.780237,2.173206,2.808225,0.804071,0.115734,0.165904,0.023834,5.056226
3,BiLSTM,4.558494,5.586267,0.224686,4.324740,5.264884,0.311329,0.233754,0.321382,0.086643,5.127877
4,LSTM,4.709926,5.571621,0.228746,4.529460,5.402700,0.274803,0.180465,0.168921,0.046057,3.831595
5,SimpleRNN,4.878542,5.721820,0.186603,4.829921,5.655501,0.205349,0.048621,0.066319,0.018746,0.996629
6,DummyMean,5.500000,6.344289,0.000000,5.509223,6.355293,-0.003472,-0.009223,-0.011004,-0.003472,-0.167689


In [ ]:
# 12. Statistical description of leakage gap across the five folds
# This is descriptive because the two validation schemes do not yield paired biological test sets.
fold_metric_rows = []

chem_fold = pd.read_csv(OUT_DIR / 'NB09A_rowlevel_fold_metrics_partial_chemometrics.csv')
fold_metric_rows.append(chem_fold)

for (fold, model, seed), g in dl.groupby(['row_fold','model','seed']):
    fold_metric_rows.append(pd.DataFrame([{
        'row_fold':fold,'model':model,'seed':seed,
        **reg_metrics(g.storage_days, g.y_pred)
    }]))

fold_all = pd.concat(fold_metric_rows, ignore_index=True)
fold_all.to_csv(OUT_DIR / 'NB09A_rowlevel_fold_seed_metrics.csv', index=False)

# Seed-mean predictions within each row fold for DL
fold_seedmean = []
for (fold, model), g in row_oof.groupby(['row_fold','model']):
    fold_seedmean.append({
        'row_fold':fold,'model':model,
        **reg_metrics(g.storage_days, g.y_pred)
    })
fold_seedmean = pd.DataFrame(fold_seedmean)
fold_seedmean.to_csv(OUT_DIR / 'NB09A_rowlevel_fold_seedmean_metrics.csv', index=False)

display(
    fold_seedmean.groupby('model')['MAE_days']
    .agg(['mean','std','min','max'])
    .sort_values('mean')
)

,mean,std,min,max
model,,,,
SVR,2.052577,0.212547,1.846579,2.383157
PLSR,2.104513,0.152362,1.938507,2.323961
ANN,2.173206,0.107096,2.010043,2.271470
BiLSTM,4.324740,0.233943,4.023690,4.650813
LSTM,4.529460,0.377334,4.199545,5.163085
SimpleRNN,4.829921,0.424457,4.423789,5.375394
DummyMean,5.509223,0.340848,4.993687,5.806617


In [ ]:
# 13. Freeze notebook run metadata
run_summary = {
    'run_revision': RUN_REVISION,
    'analysis_role': 'review-driven diagnostic; does not replace frozen NB01-NB08',
    'raw_dataset_sha256': observed_hash,
    'row_level_outer_folds': 5,
    'row_level_shuffle_seed': 2026,
    'deep_learning_seeds': SEEDS,
    'representative_configs': rep,
    'primary_valid_reference': 'frozen egg-disjoint OOF results from NB03/NB04/NB07',
    'completed': True,
}
(OUT_DIR / 'NB09A_run_summary.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')
print(json.dumps(run_summary, indent=2))
print('NB09A COMPLETED')

{
  "run_revision": "NB09A_v1_review_rowlevel_diagnostic",
  "analysis_role": "review-driven diagnostic; does not replace frozen NB01-NB08",
  "raw_dataset_sha256": "cd5021c555ae6b57f892549c574599cef75edf87f58b3f7f4d246ade9327d15e",
  "row_level_outer_folds": 5,
  "row_level_shuffle_seed": 2026,
  "deep_learning_seeds": [
    2026,
    2027,
    2028
  ],
  "representative_configs": {
    "PLSR": {
      "preprocessing": "snv",
      "n_components": 40
    },
    "SVR": {
      "preprocessing": "sg_deriv1",
      "C": 100000.0,
      "epsilon": 2.0,
      "gamma": 1e-05
    },
    "ANN": {
      "preprocessing": "sg_deriv1",
      "epochs": 75
    },
    "SimpleRNN": {
      "preprocessing": "sg_deriv1",
      "epochs": 34
    },
    "LSTM": {
      "preprocessing": "sg_deriv1",
      "epochs": 76
    },
    "BiLSTM": {
      "preprocessing": "sg_deriv1",
      "epochs": 28
    },
    "DummyMean": {}
  },
  "primary_valid_reference": "frozen egg-disjoint OOF results from NB03/NB04/NB07

In [14]:
# ============================================================
# DESCARGAR RESULTADOS NB09A EN ZIP
# ============================================================

from pathlib import Path
import shutil
from google.colab import files

RESULTS_DIR = Path(
    "/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026/"
    "05_RESULTS/NB09A_ROW_LEVEL_LEAKAGE_DIAGNOSTIC"
)

ZIP_BASE = Path("/content/NB09A_RESULTS_ROW_LEVEL_LEAKAGE_DIAGNOSTIC")

assert RESULTS_DIR.exists(), f"No existe la carpeta: {RESULTS_DIR}"

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=str(RESULTS_DIR)
)

print("✅ ZIP creado correctamente:")
print(zip_path)

files.download(zip_path)

✅ ZIP creado correctamente:
/content/NB09A_RESULTS_ROW_LEVEL_LEAKAGE_DIAGNOSTIC.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>